In [5]:
from token_reduction.model import TrainerDeepSeekMTPMerge
import os
import torch
import torch.nn as nn

In [6]:
class RoPE(nn.Module):
    # features are paired x_i, x_{i + d_head/2}
    def __init__(self, dhead, length):
        super().__init__()
        self.dhead = dhead
        self.length = length
        angle_exponents = torch.arange(0, dhead, 2) / dhead
        angles = torch.pow(1 / 10000, angle_exponents).reshape(1, -1)
        angle_per_token = angles * torch.arange(0, length).reshape(-1, 1)
        self.register_buffer("sin", torch.sin(angle_per_token).repeat(1, 2))
        self.register_buffer("cos", torch.cos(angle_per_token).repeat(1, 2))

    def forward(self, x):
        [y1, y2] = torch.chunk(x, chunks=2, dim=-1)
        x_rotated = torch.cat([-y2, y1], dim=-1)
        return x * self.cos + x_rotated * self.sin

In [ ]:
rope_layer = RoPE(12, 10)

In [7]:
batch_size = 3
seq_len = 10
h_model = 12

# Create random tensor
input_tensor = torch.randn(batch_size, seq_len, h_model)
input_tensor.shape

torch.Size([3, 10, 12])

In [ ]:
rope_layer(input_tensor).shape